# Sentiment Analysis on Tweets

This project analyzes tweets and classifies them as Positive, Negative, or Neutral.

In [ ]:
# Install Required Libraries
!pip install tweepy transformers datasets scikit-learn nltk gensim

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Download NLTK Data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

In [ ]:
# Load Dataset (from Kaggle example)
data = pd.read_csv('sentiment140.csv')  # Replace with your dataset path
tweets = data['text'].values
sentiments = data['target'].values  # 0: negative, 2: neutral, 4: positive

# Map target labels
label_map = {0: 0, 2: 1, 4: 2}
labels = [label_map[s] for s in sentiments]

In [ ]:
# Preprocessing Function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = re.sub(r'http\S+|@\S+|[^A-Za-z\s]', '', text)
    tokens = nltk.word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

processed_tweets = [preprocess(tweet) for tweet in tweets]

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(processed_tweets)

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, labels, test_size=0.2, random_state=42)

In [ ]:
# Train Logistic Regression Classifier
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Predictions and Evaluation
y_pred = lr_model.predict(X_test)
print("Logistic Regression Classification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Fine-tune BERT Model

# Prepare Hugging Face Dataset
dataset = Dataset.from_dict({'text': processed_tweets, 'label': labels})
dataset = dataset.train_test_split(test_size=0.2)

# Tokenizer and Tokenization Function
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Load Pre-trained BERT Model
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Training Arguments
training_args = TrainingArguments(
    output_dir='./bert-results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch"
)

# Initialize Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

# Train BERT
trainer.train()

In [ ]:
# Evaluate BERT
results = trainer.evaluate()
print("\nBERT Evaluation Results:\n", results)

In [ ]:
# Predict Sentiment on Custom Input
def predict_sentiment(text):
    processed = preprocess(text)
    inputs = tokenizer(processed, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    logits = outputs.logits
    sentiment = torch.argmax(logits).item()
    sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return sentiment_map[sentiment]

# Example Usage
custom_tweet = "I love how technology makes our life so much easier!"
print("\nCustom Tweet Sentiment Prediction:", predict_sentiment(custom_tweet))